# Split and Encode

This notebook splits the processed dataset into train, validation, and test sets. Encoding will be added in a later step.

## Planned Steps

- Load the processed dataset
- Drop identifier-only fields
- Create stratified train/validation/test splits
- Report split sizes and approval rates

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_PATH = Path('data/processed/featured.csv')
if not DATA_PATH.exists():
    DATA_PATH = Path('../data/processed/featured.csv')

df = pd.read_csv(DATA_PATH)
df = df.drop(columns=['ApplicationID'])

def stratified_split(frame, target_col, train_frac=0.70, val_frac=0.15, test_frac=0.15, random_state=42):
    if abs((train_frac + val_frac + test_frac) - 1.0) > 1e-9:
        raise ValueError('Split fractions must sum to 1.0')

    rng = np.random.RandomState(random_state)
    train_parts = []
    val_parts = []
    test_parts = []

    for _, group in frame.groupby(target_col, sort=False):
        idx = group.index.to_numpy().copy()
        rng.shuffle(idx)
        n = len(idx)
        n_train = int(round(n * train_frac))
        n_val = int(round(n * val_frac))
        n_test = n - n_train - n_val

        train_idx = idx[:n_train]
        val_idx = idx[n_train:n_train + n_val]
        test_idx = idx[n_train + n_val:]

        train_parts.append(frame.loc[train_idx])
        val_parts.append(frame.loc[val_idx])
        test_parts.append(frame.loc[test_idx])

    train_df = pd.concat(train_parts).sample(frac=1, random_state=random_state)
    val_df = pd.concat(val_parts).sample(frac=1, random_state=random_state)
    test_df = pd.concat(test_parts).sample(frac=1, random_state=random_state)
    return train_df, val_df, test_df

train_df, val_df, test_df = stratified_split(df, 'Approval', random_state=42)

def split_summary(name, split_df):
    return {
        'split': name,
        'row_count': int(len(split_df)),
        'approval_rate': round((split_df['Approval'].eq('Yes').mean() * 100), 2),
    }

summary = pd.DataFrame([
    split_summary('Train', train_df),
    split_summary('Validation', val_df),
    split_summary('Test', test_df),
])

print(summary.to_string(index=False))

train_path = Path('data/processed/train.csv')
val_path = Path('data/processed/validation.csv')
test_path = Path('data/processed/test.csv')
train_path.parent.mkdir(parents=True, exist_ok=True)

train_df.to_csv(train_path, index=False)
val_df.to_csv(val_path, index=False)
test_df.to_csv(test_path, index=False)

print(f'Saved train to {train_path}')
print(f'Saved validation to {val_path}')
print(f'Saved test to {test_path}')


     split  row_count  approval_rate
     Train       5600           56.0
Validation       1200           56.0
      Test       1200           56.0
Saved train to data/processed/train.csv
Saved validation to data/processed/validation.csv
Saved test to data/processed/test.csv


## One-Hot Encoding

The six categorical feature columns are one-hot encoded with `drop_first=True`. The encoder is fit on the training set and the same column set is applied to validation and test.

In [2]:
categorical_cols = ['Gender', 'MaritalStatus', 'EducationLevel', 'State', 'ResidenceType', 'EmploymentType']

def one_hot_fit_transform(train_df, val_df, test_df, categorical_cols):
    train_encoded = pd.get_dummies(train_df, columns=categorical_cols, drop_first=True)
    encoded_columns = train_encoded.columns

    def transform(frame):
        encoded = pd.get_dummies(frame, columns=categorical_cols, drop_first=True)
        encoded = encoded.reindex(columns=encoded_columns, fill_value=0)
        return encoded

    val_encoded = transform(val_df)
    test_encoded = transform(test_df)
    return train_encoded, val_encoded, test_encoded

train_encoded, val_encoded, test_encoded = one_hot_fit_transform(train_df, val_df, test_df, categorical_cols)

encoded_train_path = Path('data/processed/encoded_train.csv')
encoded_val_path = Path('data/processed/encoded_validation.csv')
encoded_test_path = Path('data/processed/encoded_test.csv')

train_encoded.to_csv(encoded_train_path, index=False)
val_encoded.to_csv(encoded_val_path, index=False)
test_encoded.to_csv(encoded_test_path, index=False)

print(f'Saved encoded train to {encoded_train_path}')
print(f'Saved encoded validation to {encoded_val_path}')
print(f'Saved encoded test to {encoded_test_path}')
print(f'Encoded train shape: {train_encoded.shape}')
print(f'Encoded validation shape: {val_encoded.shape}')
print(f'Encoded test shape: {test_encoded.shape}')
new_columns = [c for c in train_encoded.columns if c not in train_df.columns]
print('Sample new encoded column names:', new_columns[:15])


Saved encoded train to data/processed/encoded_train.csv
Saved encoded validation to data/processed/encoded_validation.csv
Saved encoded test to data/processed/encoded_test.csv
Encoded train shape: (5600, 88)
Encoded validation shape: (1200, 88)
Encoded test shape: (1200, 88)
Sample new encoded column names: ['Gender_Male', 'Gender_Others', 'MaritalStatus_Married', 'MaritalStatus_Single', 'MaritalStatus_Widowed', "EducationLevel_Bachelor's", 'EducationLevel_Doctorate', 'EducationLevel_High School', "EducationLevel_Master's", 'State_AL', 'State_AR', 'State_AZ', 'State_CA', 'State_CO', 'State_CT']
